# Practice Area and Consultant API
## A self-contained FastAPI + SQLAlchemy + Pydantic guide

This notebook explains the application from its problem statement through its files, endpoints, data flow, and testing.

- **FastAPI** receives HTTP requests and routes them to endpoint functions.
- **Pydantic** validates request data and defines response shapes.
- **SQLAlchemy ORM** maps Python classes to database tables and provides queries and updates.
- **SQLite** stores records in `practice.db`.
- **Uvicorn** runs the FastAPI application.

The examples use Pydantic v2 syntax, including `model_dump()` and `ConfigDict(from_attributes=True)`.

## 1. Application structure

Create a folder named `practice_area_api`:

```text
practice_area_api/
├── requirements.txt
├── database.py
├── models.py
├── schemas.py
└── main.py
```

| File | Responsibility | Main question it answers |
|---|---|---|
| `database.py` | Database URL, engine, session factory, shared model base, session dependency | How does the app connect to SQLite and obtain a session? |
| `models.py` | SQLAlchemy ORM classes, columns, foreign key, relationships | What is stored, and how are records connected? |
| `schemas.py` | Pydantic request/response models and validation rules | What data can enter and leave the API? |
| `main.py` | App setup, routes, queries, error handling, persistence | What happens when a client calls an endpoint? |
| `requirements.txt` | Required packages | What must be installed? |

### Request-to-database flow

```text
Client sends JSON
      |
      v
FastAPI selects the endpoint
      |
      v
Pydantic request schema validates JSON
      |
      v
main.py maps fields to a SQLAlchemy ORM object
      |
      v
SQLAlchemy session works with SQLite
      |
      v
Endpoint returns an ORM object or dictionary
      |
      v
FastAPI applies the response schema and sends JSON
```

Schemas and ORM models are different objects. The endpoint maps request fields to an ORM object. `from_attributes=True` helps Pydantic read values from an ORM object for a response.

## 2. Turn the requirements into a design

The application manages **practice areas** and **consultants**.

Example:
- Practice area: `Data Engineering`
- Floor number: `3`
- Each consultant has a name, designation, consultation fee, experience, and associated practice area.
- One practice area can have multiple consultants.

| Entity | Fields | Relationship |
|---|---|---|
| Practice area | `id`, `name`, `floor_number` | Has many consultants |
| Consultant | `id`, `name`, `designation`, `consultation_fee`, `experience_years`, `practice_area_id` | Belongs to one practice area |

| Requirement | File |
|---|---|
| Use SQLite | `database.py` |
| Define tables and relationship | `models.py` |
| Validate required fields and fee limits | `schemas.py` |
| Implement create, retrieve, filter, sort, update | `main.py` |
| List dependencies | `requirements.txt` |

The four endpoints in this version:
1. `POST /practice-areas` — create a practice area.
2. `POST /practice-areas/{practice_area_id}/consultants` — add a consultant.
3. `GET /practice-areas/{practice_area_id}` — retrieve an area with optional filtering/sorting.
4. `PUT /practice-areas/{practice_area_id}/consultants/{consultant_id}` — update a consultant within the specified area.

## 3. `requirements.txt`

Create `requirements.txt`:

In [ ]:
fastapi
uvicorn
sqlalchemy
pydantic

Install packages from the project folder:

```bash
pip install -r requirements.txt
```

If the project specifies package versions, use those versions. This notebook uses Pydantic v2 syntax.

## 4. `database.py` — connection and sessions

Create `database.py`:

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

DATABASE_URL = "sqlite:///./practice.db"

engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

Base = declarative_base()


def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

### 4.1 Database URL and engine

```python
DATABASE_URL = "sqlite:///./practice.db"
```

This selects a SQLite file named `practice.db` relative to the current working directory.

```python
engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)
```

The engine manages database connectivity. It is not a session or a table. The `check_same_thread=False` option relaxes SQLite's default thread restriction for connections; it does not make one session safe to share between simultaneous requests.

### 4.2 Session factory

```python
SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)
```

`SessionLocal` is a **factory** that creates sessions; it is not itself a session. Calling `SessionLocal()` creates a session.

- `bind=engine`: sessions use this engine.
- `autocommit=False`: changes are not automatically committed.
- `autoflush=False`: pending changes are not automatically flushed before every query; `commit()` still flushes pending changes before committing.

Remember:

```text
engine        -> manages database connectivity
SessionLocal  -> creates sessions
db            -> one session used for a unit of work
```

### 4.3 What is `Base`?

```python
Base = declarative_base()
```

`declarative_base()` creates the base class used by SQLAlchemy's declarative ORM system. Model classes inherit from this `Base`, allowing SQLAlchemy to register their table definitions and keep them in `Base.metadata`.

For example, `class PracticeArea(Base): ...` registers the practice-area table definition. The `Consultant` model registers its definition in the same metadata collection.

Think of `Base.metadata` as the collection of table definitions known to SQLAlchemy. This is why `main.py` later calls:

```python
Base.metadata.create_all(bind=engine)
```

That call asks SQLAlchemy to create missing tables. It does not automatically alter existing tables when a model changes. Defining `Base` alone does not create tables; model classes must be imported and registered before `create_all()` can see their definitions.

### 4.4 `get_db()` and dependency injection

```python
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()
```

`SessionLocal()` creates a session. `yield db` makes it available to an endpoint. FastAPI resumes the dependency generator so the cleanup code runs, and `finally` closes the session even if request handling raises an exception.

An endpoint requests the dependency using:

```python
db: Session = Depends(get_db)
```

`Depends(get_db)` tells FastAPI to provide the session. The endpoint does not call `get_db()` manually.

## 5. `models.py` — database tables and relationships

Create `models.py`:

In [ ]:
from sqlalchemy import Column, Integer, String, ForeignKey
from sqlalchemy.orm import relationship

from database import Base


class PracticeArea(Base):
    __tablename__ = "practice_areas"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False)
    floor_number = Column(Integer)

    consultants = relationship(
        "Consultant",
        back_populates="practice_area",
        cascade="all, delete-orphan"
    )


class Consultant(Base):
    __tablename__ = "consultants"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False)
    designation = Column(String, nullable=False)
    consultation_fee = Column(Integer, nullable=False)
    experience_years = Column(Integer, nullable=False)

    practice_area_id = Column(
        Integer,
        ForeignKey("practice_areas.id"),
        nullable=False
    )

    practice_area = relationship(
        "PracticeArea",
        back_populates="consultants"
    )

### Why import `Base` from `database.py`?

Both ORM models inherit from the same `Base`, so their table definitions are registered in the metadata used by `Base.metadata.create_all(...)`. Do not create a separate `Base` in this file.

### `PracticeArea` and its columns

```python
class PracticeArea(Base):
    __tablename__ = "practice_areas"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False)
    floor_number = Column(Integer)
```

The class represents a practice-area ORM object. `__tablename__` sets the database table name. `Column(...)` declares a table column. `index=True` requests an index; the primary key already guarantees uniqueness. `nullable=False` makes the column non-nullable.

### Parent-to-child relationship

```python
consultants = relationship(
    "Consultant",
    back_populates="practice_area",
    cascade="all, delete-orphan"
)
```

This lets Python navigate from a practice area to its consultants using `parent.consultants`.

- `"Consultant"` identifies the related model.
- `back_populates="practice_area"` links to the matching relationship on `Consultant`.
- `cascade="all, delete-orphan"` means deleting a practice area through the ORM also deletes associated consultants. Keep this only if that deletion behavior is intended.

`relationship()` is an ORM navigation feature; it does not create a database column.

### `Consultant` and its columns

```python
class Consultant(Base):
    __tablename__ = "consultants"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False)
    designation = Column(String, nullable=False)
    consultation_fee = Column(Integer, nullable=False)
    experience_years = Column(Integer, nullable=False)
```

These columns represent the consultant's stored details. The integer fee type follows the schema used here. In financial systems, `Numeric`/`Decimal` is often preferable to floating-point values.

### Foreign-key column and reverse relationship

```python
practice_area_id = Column(
    Integer,
    ForeignKey("practice_areas.id"),
    nullable=False
)
```

This is the database-level link to the parent table.

```python
practice_area = relationship(
    "PracticeArea",
    back_populates="consultants"
)
```

This is the ORM-level link back to the parent object.

- `practice_area_id` is a stored database column.
- `practice_area` is a Python relationship attribute.
- `PracticeArea.consultants` gives access from parent to children.

Example records:

```text
practice_areas
id | name             | floor_number
1  | Data Engineering | 3

consultants
id | name  | consultation_fee | practice_area_id
7  | Rahul | 500              | 1
```

The consultant's `practice_area_id` links it to the practice area whose ID is `1`.

## 6. `schemas.py` — input validation and output shape

Create `schemas.py`:

In [ ]:
from pydantic import BaseModel, ConfigDict, Field


class PracticeAreaCreate(BaseModel):
    name: str = Field(min_length=1)
    floor_number: int = Field(ge=1)


class ConsultantCreate(BaseModel):
    name: str = Field(min_length=1)
    designation: str = Field(min_length=1)
    consultation_fee: int = Field(ge=300)
    experience_years: int = Field(ge=0)


class ConsultantUpdate(BaseModel):
    name: str | None = Field(default=None, min_length=1)
    designation: str | None = Field(default=None, min_length=1)
    consultation_fee: int | None = Field(default=None, ge=300)
    experience_years: int | None = Field(default=None, ge=0)


class ConsultantOut(BaseModel):
    id: int
    name: str
    designation: str
    consultation_fee: int
    experience_years: int

    model_config = ConfigDict(from_attributes=True)


class PracticeAreaOut(BaseModel):
    id: int
    name: str
    floor_number: int

    model_config = ConfigDict(from_attributes=True)


class PracticeAreaWithConsultants(PracticeAreaOut):
    consultants: list[ConsultantOut]

### Why schemas are separate from ORM models

ORM models describe what is stored. Schemas describe what the API accepts and returns.

The database generates the practice-area ID, so the create request usually needs only `name` and `floor_number`. The response includes the generated ID.

### Create schemas and validation

`PracticeAreaCreate` validates a practice-area request. `ConsultantCreate` validates a consultant request.

- `min_length=1` requires at least one character.
- `ge=1` means greater than or equal to one.
- `ge=300` means the fee must be at least 300.
- `ge=0` means experience cannot be negative.

FastAPI validates the request body before the endpoint's main logic runs. Invalid values typically produce HTTP `422`.

### Why a separate `ConsultantUpdate`?

Each field has `default=None`, allowing the client to send only the fields it wants to change. For example:

```json
{"consultation_fee": 600}
```

This differs from `ConsultantCreate`, where all required creation fields must be present.

The update endpoint later uses `model_dump(exclude_unset=True)` to include only fields actually supplied. If a client explicitly sends a field as `null`, it is still considered supplied; validation/update logic must handle that appropriately.

### Output schemas and `from_attributes=True`

```python
model_config = ConfigDict(from_attributes=True)
```

This is Pydantic v2 configuration. It allows Pydantic to read values from an object's attributes rather than requiring a dictionary.

A SQLAlchemy query returns an ORM object with attributes like `consultant.name`. When the endpoint returns that object and declares `response_model=ConsultantOut`, FastAPI/Pydantic can read those attributes and serialize the response.

This setting does not save data or map request fields into ORM objects. The endpoint performs request-to-ORM mapping; `from_attributes=True` supports ORM-to-response mapping.

### Nested response and inheritance

```python
class PracticeAreaWithConsultants(PracticeAreaOut):
    consultants: list[ConsultantOut]
```

This inherits `id`, `name`, and `floor_number` from `PracticeAreaOut` and adds a list of consultant output objects.

The schema defines the response shape; the endpoint must retrieve and provide the related consultants.

## 7. `main.py` — imports and app setup

Create `main.py`:

In [ ]:
from fastapi import Depends, FastAPI, HTTPException, Query
from sqlalchemy.orm import Session

from database import Base, engine, get_db
from models import Consultant, PracticeArea
from schemas import (
    ConsultantCreate,
    ConsultantOut,
    PracticeAreaCreate,
    PracticeAreaOut,
    ConsultantUpdate,
    PracticeAreaWithConsultants,
)

Base.metadata.create_all(bind=engine)
app = FastAPI(title="Practice Area and Consultant API")

`FastAPI` creates the app. `Depends` supplies dependencies. `HTTPException` returns errors. `Query` defines query parameters. `Session` annotates the database session.

The models and schemas are imported from their own files. Model imports happen before `create_all()` so their table definitions are registered in `Base.metadata`.

```python
Base.metadata.create_all(bind=engine)
```

This asks SQLAlchemy to create missing tables. It does not migrate existing tables when their definitions change.

```python
app = FastAPI(title="Practice Area and Consultant API")
```

The title appears in the interactive docs at `/docs`.

## 8. Endpoint 1 — create a practice area

Design:
- Method: `POST`
- Path: `/practice-areas`
- Input schema: `PracticeAreaCreate`
- ORM model: `PracticeArea`
- Response schema: `PracticeAreaOut`
- Success status: `201 Created`

Code:

In [ ]:
@app.post("/practice-areas", response_model=PracticeAreaOut, status_code=201)
def create_practice_area(
    practice_area: PracticeAreaCreate,
    db: Session = Depends(get_db),
):
    item = PracticeArea(**practice_area.model_dump())
    db.add(item)
    db.commit()
    db.refresh(item)
    return item

The decorator registers the POST route. `response_model` defines the response shape, and `status_code=201` indicates successful creation.

`practice_area: PracticeAreaCreate` means FastAPI validates the JSON body using that schema. `db: Session = Depends(get_db)` asks FastAPI for a session.

### Mapping schema data into an ORM object

```python
item = PracticeArea(**practice_area.model_dump())
```

`model_dump()` turns the Pydantic object into a dictionary, such as:

```python
{"name": "Data Engineering", "floor_number": 3}
```

`**` unpacks the dictionary into keyword arguments. It is equivalent to:

```python
item = PracticeArea(name="Data Engineering", floor_number=3)
```

The values come from the request.

Then:
- `db.add(item)` stages the object.
- `db.commit()` commits the transaction.
- `db.refresh(item)` reloads generated values, including the new ID.
- `return item` lets FastAPI format the object with `PracticeAreaOut`.

Pattern: **validate -> construct ORM object -> add -> commit -> refresh -> return**.

## 9. Endpoint 2 — add a consultant

Design:
- Method: `POST`
- Path: `/practice-areas/{practice_area_id}/consultants`
- Path parameter: parent ID
- Input schema: `ConsultantCreate`
- ORM model: `Consultant`
- Response schema: `ConsultantOut`

Code:

In [ ]:
@app.post(
    "/practice-areas/{practice_area_id}/consultants",
    response_model=ConsultantOut,
    status_code=201,
)
def add_consultant(
    practice_area_id: int,
    consultant: ConsultantCreate,
    db: Session = Depends(get_db),
):
    parent = db.get(PracticeArea, practice_area_id)
    if parent is None:
        raise HTTPException(status_code=404, detail="Practice area not found")

    item = Consultant(
        **consultant.model_dump(),
        practice_area_id=practice_area_id,
    )
    db.add(item)
    db.commit()
    db.refresh(item)
    return item

The path ID identifies the parent practice area; the body describes the consultant.

```python
parent = db.get(PracticeArea, practice_area_id)
```

`Session.get()` retrieves an ORM object by primary-key value or returns `None`. If no parent exists, the endpoint raises `404` and stops.

```python
item = Consultant(
    **consultant.model_dump(),
    practice_area_id=practice_area_id,
)
```

The dictionary contains validated consultant fields. `**` unpacks them into the `Consultant` constructor. The parent ID is added separately because it came from the URL.

The endpoint then adds, commits, refreshes, and returns the consultant using `ConsultantOut`.

Example request body:

```json
{
  "name": "Rahul",
  "designation": "Senior Consultant",
  "consultation_fee": 500,
  "experience_years": 5
}
```

## 10. Endpoint 3 — retrieve, filter, and sort consultants

Design:
- Method: `GET`
- Path: `/practice-areas/{practice_area_id}`
- Optional query parameters: `designation`, `sort`
- Response schema: `PracticeAreaWithConsultants`

Code:

In [ ]:
@app.get(
    "/practice-areas/{practice_area_id}",
    response_model=PracticeAreaWithConsultants,
)
def get_practice_area(
    practice_area_id: int,
    designation: str | None = Query(default=None),
    sort: str | None = Query(default=None),
    db: Session = Depends(get_db),
):
    parent = db.get(PracticeArea, practice_area_id)
    if parent is None:
        raise HTTPException(status_code=404, detail="Practice area not found")

    consultants = list(parent.consultants)

    if designation is not None and designation.strip():
        consultants = [c for c in consultants if c.designation == designation]

    if sort is not None:
        if sort not in ("name", "consultation_fee"):
            raise HTTPException(status_code=404, detail="Invalid sort field")
        consultants.sort(key=lambda c: getattr(c, sort))

    return {
        "id": parent.id,
        "name": parent.name,
        "floor_number": parent.floor_number,
        "consultants": consultants,
    }

### Optional query parameters

```python
designation: str | None = Query(default=None)
sort: str | None = Query(default=None)
```

Both parameters may be omitted and then default to `None`.

Example:

```text
GET /practice-areas/1?designation=Senior%20Consultant&sort=name
```

### Find the parent and access related consultants

The endpoint looks up the practice area and returns `404` if missing.

```python
consultants = list(parent.consultants)
```

`parent.consultants` comes from the ORM relationship in `models.py`. It provides the related consultant objects. `list()` gives the endpoint a list to filter and sort.

### Filter by designation

```python
if designation is not None and designation.strip():
    consultants = [c for c in consultants if c.designation == designation]
```

This runs only when a non-blank designation is supplied. The list comprehension keeps consultants whose designation exactly matches it.

### Validate and apply sorting

```python
if sort is not None:
    if sort not in ("name", "consultation_fee"):
        raise HTTPException(status_code=404, detail="Invalid sort field")
    consultants.sort(key=lambda c: getattr(c, sort))
```

Only `name` and `consultation_fee` are allowed.

`lambda c: getattr(c, sort)` reads the attribute named by `sort`. If `sort` is `"name"`, it reads `c.name`; if `"consultation_fee"`, it reads that field. Without `reverse=True`, the list is sorted ascending.

This code returns `404` for an invalid sort field. Other APIs often use `400` or `422` for invalid parameters, but preserve the specified contract if one is required.

### Assemble the response

```python
return {
    "id": parent.id,
    "name": parent.name,
    "floor_number": parent.floor_number,
    "consultants": consultants,
}
```

The endpoint returns the practice-area attributes plus the filtered/sorted list. `PracticeAreaWithConsultants` defines the response shape; the endpoint performs the retrieval, filtering, sorting, and assembly.

## 11. Endpoint 4 — update a consultant

Design:
- Method: `PUT`
- Path: `/practice-areas/{practice_area_id}/consultants/{consultant_id}`
- Request schema: `ConsultantUpdate`
- Response schema: `ConsultantOut`

Code:

In [ ]:
@app.put(
    "/practice-areas/{practice_area_id}/consultants/{consultant_id}",
    response_model=ConsultantOut
)
def update_consultant(
    practice_area_id: int,
    consultant_id: int,
    data: ConsultantUpdate,
    db: Session = Depends(get_db),
):
    consultant = db.query(Consultant).filter(
        Consultant.id == consultant_id,
        Consultant.practice_area_id == practice_area_id
    ).first()

    if consultant is None:
        raise HTTPException(
            status_code=404,
            detail="Consultant not found in this practice area"
        )

    update_data = data.model_dump(exclude_unset=True)

    for field, value in update_data.items():
        setattr(consultant, field, value)

    db.commit()
    db.refresh(consultant)

    return consultant

### Why check both IDs?

```python
consultant = db.query(Consultant).filter(
    Consultant.id == consultant_id,
    Consultant.practice_area_id == practice_area_id
).first()
```

This finds a consultant only when both the consultant ID and parent practice-area ID match. It prevents updating a consultant through the URL of a different practice area.

If there is no match, the query returns `None` and the endpoint raises `404`.

### Why use `ConsultantUpdate`?

It allows a client to send only the fields it wants to change, such as:

```json
{"name": "Rahul Sharma"}
```

### `exclude_unset=True`

```python
update_data = data.model_dump(exclude_unset=True)
```

This creates a dictionary containing only fields the client actually supplied. For the example above:

```python
{"name": "Rahul Sharma"}
```

Omitted fields are not included, so they remain unchanged. An explicitly supplied `null` is still considered supplied and must be handled appropriately.

### `setattr()`

```python
for field, value in update_data.items():
    setattr(consultant, field, value)
```

For example, `setattr(consultant, "name", "Rahul Sharma")` is equivalent to `consultant.name = "Rahul Sharma"`.

Here, `data` is the Pydantic request object. `consultant` is the SQLAlchemy ORM object retrieved from the database.

Finally, `commit()` saves changes to SQLite, `refresh()` reloads the object, and the response is shaped by `ConsultantOut`.

This endpoint uses `PUT` with partial-update behavior because that is how this API is implemented. Follow the required HTTP method and behavior in the API specification.

## 12. Run the application

At the end of `main.py`, this block lets the script start Uvicorn when run directly:

In [ ]:
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8080)

Run directly with:

```bash
python main.py
```

Or run from the project directory using the common development command:

```bash
uvicorn main:app --reload --port 8080
```

In the second command, Uvicorn imports `main.py` and uses its `app` object, so the `if __name__ == "__main__"` block does not run during that import.

Open:

```text
http://127.0.0.1:8080/docs
```

## 13. Test in dependency order

### Step 1 — create a practice area

`POST /practice-areas`

```json
{
  "name": "Data Engineering",
  "floor_number": 3
}
```

Record the returned ID.

### Step 2 — add a consultant

`POST /practice-areas/{practice_area_id}/consultants`

Replace the path ID with the actual ID.

```json
{
  "name": "Rahul",
  "designation": "Senior Consultant",
  "consultation_fee": 500,
  "experience_years": 5
}
```

### Step 3 — retrieve the practice area

`GET /practice-areas/{practice_area_id}`

Check that the response contains the practice area and consultant list.

### Step 4 — filter and sort

Example:

```text
GET /practice-areas/1?designation=Senior%20Consultant&sort=consultation_fee
```

Replace `1` with the actual ID. This filters by designation and sorts the remaining consultants by fee ascending.

### Step 5 — update a consultant

`PUT /practice-areas/{practice_area_id}/consultants/{consultant_id}`

```json
{
  "name": "Rahul Sharma"
}
```

Use actual IDs, then call the GET endpoint again to verify the change.

### Useful negative tests

- Fee below `300` should fail schema validation with `422`.
- A nonexistent practice area should return `404`.
- An invalid sort field follows the endpoint's error behavior.
- A consultant not belonging to the specified practice area should return `404`.
- Malformed JSON fails before endpoint logic executes.

## 14. A reusable approach for a new API problem

### A. Identify entities
Find the main things the application stores. Each may need an ORM model.

### B. List fields
Record each field and identify required, optional, and constrained values.

### C. Draw relationships
Determine parent/child entities, where foreign-key columns belong, and which ORM relationships are needed.

### D. Design schemas
Decide what is required to create each entity, what can be changed during an update, and what each response must contain. Add nested output schemas when responses include related records.

### E. List endpoints
For every requirement, note:
- HTTP method and URL
- Path parameters
- Query parameters
- Request schema
- ORM models used
- Response schema
- Missing-record and validation cases

### F. Implement and test in dependency order
1. `requirements.txt`
2. `database.py`
3. `models.py`
4. `schemas.py`
5. `main.py`
6. Run and test each endpoint

Follow any exact file names, routes, versions, and response requirements provided by the specification.

### Common errors

| Symptom | Check first |
|---|---|
| `ModuleNotFoundError` | Package installation and import names |
| Tables missing | Model imports and `Base.metadata.create_all()` |
| `422` JSON parsing error | Exact request body |
| `422` validation error | Schema fields and constraints |
| Unexpected `404` | IDs and query filters |
| New ID not visible | Commit and refresh sequence |
| Related records missing | Foreign key, ORM relationship, response schema |
| Existing table has old columns | `create_all()` does not migrate existing tables |

### Key distinctions

- `Base` registers model table definitions in shared metadata.
- `engine` manages database connectivity.
- `SessionLocal` creates sessions.
- `get_db()` provides a session and closes it after request handling.
- ORM models describe stored records and relationships.
- Pydantic schemas validate input and define output.
- `model_dump()` converts a Pydantic model to a dictionary.
- `from_attributes=True` lets Pydantic read values from ORM object attributes.
- `exclude_unset=True` keeps only fields explicitly supplied for an update.
- `commit()` persists changes; `refresh()` reloads the ORM object.

The overall path is: request JSON → schema validation → ORM mapping → database session → response schema → response JSON.